In [ ]:
import pandas as pd
import numpy as np

import xgboost as xgb
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


In [2]:
df = pd.read_csv("/kaggle/input/property-val-dataset/train.csv")
df['date'] = pd.to_datetime(df['date'])
df['sale_year'] = df['date'].dt.year
df['sale_month'] = df['date'].dt.month
df['day_of_month'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.dayofweek

df['effective_built_year'] = df.apply(lambda x: x['yr_renovated'] if x['yr_renovated'] > 0 else x['yr_built'], axis=1)
df['house_age'] = df['sale_year'] - df['effective_built_year']

zip_map = df.groupby('zipcode').apply(lambda x: x['price'].median()).to_dict()
df['zip_index'] = df['zipcode'].map(zip_map)

/tmp/ipykernel_17/989552771.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  zip_map = df.groupby('zipcode').apply(lambda x: x['price'].median()).to_dict()


In [3]:
X = df.drop(columns=['id', 'date','zipcode',"price"])
y = df['price']

# Convert once
dtrain = xgb.DMatrix(X.values, label=y.values)